# Logit lens on GPT-2 — examples

Project the residual stream after every layer through the final LayerNorm and unembedding,
and look at the top tokens at each position — to see at what depth the model settles on its prediction.

This notebook is for looking at individual prompts. The run over all of `data/evaluations` with metrics and
conclusions is in `logit_lens_dataset.ipynb`; the same pipeline with the Jacobian lens is in `jacobian_lens.ipynb`.

## Setup

In [ ]:
import os
import sys

REPO_URL = "https://github.com/ai360-project-school-j-lens/jacobian-lens-gpt2.git"
REPO_DIR = os.path.abspath("jacobian-lens-gpt2")

if os.path.isfile("gpt2/lens_eval.py"):  # ноутбук открыт прямо из репозитория
    REPO_DIR = os.getcwd()
elif os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.append(REPO_DIR)

In [ ]:
from functools import partial

import torch
from IPython.display import display

from gpt2 import GPT2LensModel
from gpt2.lens_eval import (
    DATASETS,
    load_eval,
    logit_lens,
    readout_position,
    show_lens,
    top_tokens_table,
)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

gpt = GPT2LensModel.from_pretrained(device=DEVICE)
logit = partial(logit_lens, gpt)
gpt

## Dataset

Six prompt sets from `data/evaluations` (see its README): every item has a `prompt`, the `intermediates` —
words the model should "think about" on the way to the answer — and sometimes the `target` itself.

In [ ]:
evals = {dataset: load_eval(REPO_DIR, dataset) for dataset in DATASETS}
{dataset: len(items) for dataset, items in evals.items()}

In [ ]:
{dataset: items[0] for dataset, items in evals.items()}

## Sanity checks

Shapes of the encoded prompt and the hidden states, the lens prediction at a single position,
and the last row of the lens is exactly the model's own logits.

In [ ]:
ids = gpt.encode(evals["order-ops"][0]["prompt"])
with torch.no_grad():
    hidden_states = gpt.forward(ids, output_hidden_states=True).hidden_states

In [ ]:
ids.shape

In [ ]:
len(hidden_states), hidden_states[0].shape

In [ ]:
gpt.unembed(hidden_states[0][0, 1, :]).argmax()

In [ ]:
prompt = evals["order-ops"][0]["prompt"].rstrip()
logits = logit(prompt)
with torch.no_grad():
    model_logits = gpt._hf_model(gpt.encode(prompt)).logits[0]

logits.shape, torch.allclose(logits[-1], model_logits, atol=1e-3)

## Logit lens visualisation

Handwritten logit lens (`gpt2/lens_eval.py: logit_lens`): the output of every block (`hidden_states` from `gpt.forward`)
goes through `ln_f` + unembedding; the last row is the model output.
`show_lens` packs the result into `SliceData` by hand and renders it with `build_page` from `jlens.vis`:
an interactive position × layer view of token ranks.

### Dataset prompts

In [ ]:
for sample in evals["order-ops"][0:5]:
    show_lens(
        gpt,
        logit,
        sample["prompt"].rstrip(),
        title=sample["name"],
        description=f"target: {sample.get('target')!r}, intermediates: {sample['intermediates']}",
    )

### One prompt from every dataset

The interesting cell is the readout position from `data/evaluations/README.md`: the last prompt token
(for poetry — the newline at the end of the first line of the couplet).

In [ ]:
for dataset, items in evals.items():
    sample = items[0]
    show_lens(
        gpt,
        logit,
        sample["prompt"].rstrip(),
        title=f"{dataset}: {sample['name']}",
        description=f"target: {sample.get('target')!r}, intermediates: {sample['intermediates']}",
    )

### Top-5 at the readout position, layer by layer

In [ ]:
for dataset in ["multihop", "multilingual", "poetry", "association"]:
    sample = evals[dataset][0]
    prompt = sample["prompt"].rstrip()
    position = readout_position(gpt.tokenizer, gpt.encode(prompt)[0].tolist(), dataset)
    print(f"{dataset}: {prompt!r}\nintermediates: {sample['intermediates']}, target: {sample.get('target')!r}")
    display(top_tokens_table(gpt, {"logit lens": logit}, prompt, position=position))

### Custom prompt

In [ ]:
custom_prompt = "The Eiffel Tower is located in the city of"
show_lens(gpt, logit, custom_prompt)
top_tokens_table(gpt, {"logit lens": logit}, custom_prompt)